# Clase 123 — PyTorch Lightning Trainer

Lightning organiza tu código en `LightningModule` + `LightningDataModule` + `Trainer`. Soporta DDP, FSDP, mixed precision, checkpoints y logging automático.

Fallback completo si `lightning`/`torch` no están.

In [ ]:
USE_PL = False
try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import DataLoader, TensorDataset
    import pytorch_lightning as pl
    USE_PL = True
    print('lightning:', pl.__version__)
except Exception as e:
    print('lightning no disponible. Fallback con sklearn MLP. Motivo:', type(e).__name__)
import numpy as np
np.random.seed(42)

## 1. Dataset sintético (clasificación 3 clases)

In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
X, y = make_classification(n_samples=2000, n_features=20, n_informative=10, n_classes=3, n_clusters_per_class=2, random_state=42)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)
print('train:', Xtr.shape, '| test:', Xte.shape)

## 2. LightningModule + LightningDataModule

In [ ]:
if USE_PL:
    class MLP(pl.LightningModule):
        def __init__(self, in_dim=20, hid=64, n_classes=3, lr=1e-3):
            super().__init__(); self.save_hyperparameters()
            self.net = nn.Sequential(nn.Linear(in_dim, hid), nn.ReLU(), nn.Linear(hid, hid), nn.ReLU(), nn.Linear(hid, n_classes))
        def forward(self, x): return self.net(x)
        def training_step(self, batch, _):
            x, y = batch; logits = self(x); loss = F.cross_entropy(logits, y)
            self.log('train_loss', loss); return loss
        def validation_step(self, batch, _):
            x, y = batch; logits = self(x); loss = F.cross_entropy(logits, y)
            acc = (logits.argmax(1) == y).float().mean()
            self.log('val_loss', loss); self.log('val_acc', acc)
        def configure_optimizers(self):
            return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)

    class DM(pl.LightningDataModule):
        def setup(self, stage=None):
            self.tr = TensorDataset(torch.tensor(Xtr, dtype=torch.float32), torch.tensor(ytr))
            self.te = TensorDataset(torch.tensor(Xte, dtype=torch.float32), torch.tensor(yte))
        def train_dataloader(self): return DataLoader(self.tr, batch_size=64, shuffle=True)
        def val_dataloader(self): return DataLoader(self.te, batch_size=64)
    print('Lightning module + data module definidos')
else:
    print('Fallback: usaremos sklearn MLPClassifier en su lugar')

## 3. Trainer + entrenamiento

In [ ]:
if USE_PL:
    from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
    cbs = [
        EarlyStopping(monitor='val_loss', patience=3, mode='min'),
        ModelCheckpoint(monitor='val_acc', mode='max', save_top_k=1, filename='best-{epoch}-{val_acc:.3f}'),
    ]
    trainer = pl.Trainer(max_epochs=3, accelerator='cpu', callbacks=cbs, enable_progress_bar=False, logger=False)
    model = MLP(); dm = DM()
    trainer.fit(model, datamodule=dm)
    print('train done.')
else:
    from sklearn.neural_network import MLPClassifier
    clf = MLPClassifier(hidden_layer_sizes=(64,64), max_iter=20, random_state=42, early_stopping=True, validation_fraction=0.2).fit(Xtr, ytr)
    print(f'sklearn fallback acc: {clf.score(Xte, yte):.4f}')

## 4. Callbacks: EarlyStopping + ModelCheckpoint

- **EarlyStopping**: corta si `val_loss` no mejora `patience` epochs.
- **ModelCheckpoint**: guarda el mejor checkpoint por una métrica.
- Otros: `LearningRateMonitor`, `StochasticWeightAveraging`, `GradientAccumulationScheduler`.

## 5. DDP / distribuido (conceptual)

```python
# Multi-GPU en una sola máquina
trainer = pl.Trainer(accelerator='gpu', devices=4, strategy='ddp')

# Multi-nodo (cluster SLURM)
trainer = pl.Trainer(accelerator='gpu', devices=8, num_nodes=4, strategy='ddp')

# FSDP para modelos grandes (sharded params)
trainer = pl.Trainer(strategy='fsdp', precision='bf16-mixed')
```

Lightning maneja `init_process_group`, `DistributedSampler`, gradient sync automáticamente.

## Conclusiones

- LightningModule = bundle (modelo + loss + optimizer + steps) reusable.
- `Trainer.fit()` es agnostic al hardware: cambia `accelerator/devices/strategy` sin tocar el modelo.
- Callbacks reemplazan boilerplate (early stop, checkpoint, lr schedule).
- Para LLMs grandes, mirar FSDP/DeepSpeed integration.

## ✅ Soluciones de los ejercicios

PyTorch Lightning: `LightningModule`, callbacks, mixed precision, logging W&B y DDP. Se validan por AST sin `lightning`. El foco es *cuánto boilerplate elimina el `Trainer`* respecto al loop manual de la clase 121.

**Ej. 1 — LightningModule básico.** Portar el MLP a Lightning con `training_step`/`validation_step`.

In [ ]:
import torch
import torch.nn as nn
import lightning as L

class LitMLP(L.LightningModule):
    def __init__(self, lr=1e-3):
        super().__init__()
        self.save_hyperparameters()
        self.net = nn.Sequential(nn.Flatten(), nn.Linear(784, 128),
                                 nn.ReLU(), nn.Linear(128, 10))

    def training_step(self, batch, _):
        x, y = batch
        loss = nn.functional.cross_entropy(self.net(x), y)
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, _):
        x, y = batch
        self.log("val_loss", nn.functional.cross_entropy(self.net(x), y), prog_bar=True)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)

print("MLP portado a LightningModule (save_hyperparameters guarda lr en el checkpoint)")

**Ej. 2 — Callbacks.** `EarlyStopping(patience=5)` + `ModelCheckpoint(save_top_k=3)`.

In [ ]:
import lightning as L
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint

callbacks = [
    EarlyStopping(monitor="val_loss", patience=5, mode="min"),
    ModelCheckpoint(monitor="val_loss", save_top_k=3, mode="min",
                    filename="{epoch}-{val_loss:.2f}"),
]
# trainer = L.Trainer(max_epochs=50, callbacks=callbacks)
print("EarlyStopping corta si val_loss no mejora en 5 epochs; guarda los 3 mejores checkpoints")

**Ej. 3 — Mixed precision.** `precision='bf16-mixed'`: ~2x throughput, sin loss-scaling.

In [ ]:
import lightning as L

trainer = L.Trainer(max_epochs=10, precision="bf16-mixed")   # bf16 en GPUs Ampere+
# bf16 tiene el mismo rango que fp32 -> no necesita loss-scaling (a diferencia de fp16).
print("precision='bf16-mixed': casi 2x mas rapido y menos VRAM en GPUs modernas")

**Ej. 4 — W&B logging.** `WandbLogger(project='test')` para ver las curvas online.

In [ ]:
import lightning as L
from lightning.pytorch.loggers import WandbLogger

logger = WandbLogger(project="test")     # requiere `wandb login` una vez
# trainer = L.Trainer(max_epochs=10, logger=logger)
# todo lo que pases a self.log(...) aparece en el dashboard de Weights & Biases.
print("WandbLogger: metricas y curvas en el dashboard online de W&B")

**Ej. 5 — DDP.** `strategy='ddp', devices=2`: una copia por GPU, gradientes por all-reduce.

In [ ]:
import lightning as L

trainer = L.Trainer(strategy="ddp", devices=2, accelerator="gpu", max_epochs=10)
# DDP: cada GPU tiene una replica del modelo y su shard del batch; los gradientes se
# promedian con all-reduce. Speedup ~ lineal si el batch por GPU se mantiene.
print("strategy='ddp', devices=2: data-parallel en 2 GPUs")